In [14]:
import heapq

def get_user_graph(n, matrix):
    """
    n: number of vertices
    matrix: adjacency matrix as a list of lists
    """
    return matrix, n

# MST heuristic (Prim’s algorithm for subset of unvisited nodes)
def mst_heuristic(adj, unvisited):
    if not unvisited:
        return 0
    visited = set()
    heap = []
    start = next(iter(unvisited))
    visited.add(start)

    for v, cost in enumerate(adj[start]):
        if v in unvisited and cost > 0:
            heapq.heappush(heap, (cost, start, v))

    mst_cost = 0
    while heap and len(visited) < len(unvisited):
        cost, u, v = heapq.heappop(heap)
        if v not in visited:
            mst_cost += cost
            visited.add(v)
            for w, next_cost in enumerate(adj[v]):
                if w in unvisited and w not in visited and next_cost > 0:
                    heapq.heappush(heap, (next_cost, v, w))
    if len(visited) < len(unvisited):
        return float('inf')
    return mst_cost

# State class for A*
class State:
    def __init__(self, path, visited, cost, heuristic_cost, start):
        self.path = path
        self.visited = visited
        self.cost = cost
        self.heuristic_cost = heuristic_cost
        self.start = start

    def __lt__(self, other):
        return (self.cost + self.heuristic_cost) < (other.cost + other.heuristic_cost)

# A* Hamiltonian cycle with MST heuristic
def a_star_hamiltonian_cycle_mst(adj, n, start=0):
    initial = State([start], set([start]), 0, 0, start)
    heap = []
    heapq.heappush(heap, initial)

    while heap:
        current = heapq.heappop(heap)

        # Goal check: all visited and edge back to start
        if len(current.visited) == n and adj[current.path[-1]][start] > 0:
            return current.path + [start], current.cost + adj[current.path[-1]][start]

        unvisited = set(range(n)) - current.visited
        for v in unvisited:
            if adj[current.path[-1]][v] > 0:
                new_path = current.path + [v]
                new_visited = set(current.visited)
                new_visited.add(v)
                new_cost = current.cost + adj[current.path[-1]][v]

                # Heuristic = MST of remaining + min edge to start + min edge from last
                remaining = unvisited - {v}
                mst_cost = mst_heuristic(adj, remaining)
                min_to_start = min([adj[v2][start] for v2 in remaining] + [adj[v][start]]) if remaining else adj[v][start]
                min_from_last = min([adj[v][v2] for v2 in remaining] + [float('inf')]) if remaining else 0
                heuristic_cost = mst_cost + min_to_start + min_from_last

                heapq.heappush(heap, State(new_path, new_visited, new_cost, heuristic_cost, start))

    return None, None

n = 4
matrix = [
    [0, 10, 1, 12],
    [10, 0, 5, 2 ],
    [15, 0, 0, 3],
    [2, 5, 10, 0]
]

adj, n = get_user_graph(n, matrix)
path, cost = a_star_hamiltonian_cycle_mst(adj, n)

if path:
    print("Hamiltonian cycle found with cost", cost, ":", path)
else:
    print("No Hamiltonian cycle exists in the given graph.")


Hamiltonian cycle found with cost 19 : [0, 2, 3, 1, 0]
